# 🔄 Notebook 3: Polling Publisher vs Change Data Capture (CDC)

The polling design works but it adds latency and load on the OLTP database. **CDC** instead reads the database's own write-ahead log (WAL) to discover changes — no polling, near-real-time.

Postgres exposes WAL changes via **logical replication slots** (the same mechanism Debezium uses).


## 🛠️ Setup (logical decoding requires `wal_level=logical`)

Our `docker-compose.yml` already starts Postgres with `wal_level=logical`:

```bash
cd 04-patterns/outbox-and-cdc
docker compose up -d
uv sync
```


## 🟩 Stream WAL changes from a logical replication slot

In [ ]:
import psycopg
DSN = 'host=localhost port=5432 user=demo password=demo dbname=outbox_demo'

with psycopg.connect(DSN, autocommit=True) as conn:
    # Drop existing slot if any (idempotent setup)
    conn.execute("SELECT pg_drop_replication_slot('demo_slot') WHERE EXISTS (SELECT 1 FROM pg_replication_slots WHERE slot_name='demo_slot')")
    conn.execute("SELECT pg_create_logical_replication_slot('demo_slot', 'test_decoding')")
    print('slot created')


In [ ]:
# Make some changes that the slot will capture
with psycopg.connect(DSN, autocommit=True) as conn:
    conn.execute("DROP TABLE IF EXISTS products")
    conn.execute("CREATE TABLE products (id SERIAL PRIMARY KEY, name TEXT, price INTEGER)")
    conn.execute("INSERT INTO products(name,price) VALUES ('book',25),('pen',5)")
    conn.execute("UPDATE products SET price=30 WHERE name='book'")
    conn.execute("DELETE FROM products WHERE name='pen'")


In [ ]:
# Drain the slot — these are the WAL events Debezium would emit to Kafka
with psycopg.connect(DSN, autocommit=True) as conn:
    rows = conn.execute("SELECT lsn, data FROM pg_logical_slot_get_changes('demo_slot', NULL, NULL)").fetchall()
for lsn, data in rows:
    print(f'{lsn}  {data}')


Each INSERT / UPDATE / DELETE produced a structured event — straight from the storage engine, **without** the application doing anything special. That's CDC.

## 📊 Polling vs CDC

| | Polling outbox | Logical CDC |
|---|---|---|
| Application code | small `INSERT` into outbox | none — DB does it |
| Latency | poll interval (100 ms – 1 s) | milliseconds |
| DB load | repeated reads of outbox | continuous WAL streaming |
| Operational complexity | one cron / worker | replication slot, lag monitoring |
| Cross-engine portability | runs anywhere SQL runs | DB-specific (WAL format) |

Many teams start with the **outbox** for simplicity and graduate to **CDC** (Debezium) once they need lower latency or a richer event stream.